In [ ]:
"""
The purpose of this Jupyter notebook is to classify each of the
confirmed/reliable VACV WR host factors as either "proviral",
"antiviral" or None (which represents host factors with ambiguous
roles).
"""

'\nThe purpose of this Jupyter notebook is to classify each of the\nconfirmed/reliable VACV WR host factors as either proviral, antiviral or\nambiguous.\n'

In [1]:
import pandas as pd

In [2]:
# Create a dictionary mapping the confirmed host factors to their role
# ("proviral", "antiviral" or None)
hf_role_dict = {
    "proviral": [
        "EXT1", "EXT2", "EXTL3", "XYLT2", "B4GALT7", "B3GALT6",
        "B3GAT3", "NDST1", "HS2ST1", "GLCE", "UGDH", "UXS1", "SLC35B2",
        "PTAR1", "TM9SF2", "COG1", "COG2", "COG3", "COG4", "COG5",
        "COG6", "COG7", "COG8", "TMED10", "COPA", "COPB1", "COPB2",
        "COPG1", "SEC23A", "SEC23B", "SEC24A", "SEC24B", "ACTB",
        "ACTR2", "ACTR3", "ARPC1A", "ARPC2", "ARPC3", "ARPC4", "ARPC5",
        "CDC42", "RAC1", "PAK1", "PFN1", "RACGAP1", "AXL", "ANXA2",
        "GAS6", "EGFR", "LAMA1", "LAMC2", "SELPLG", "EIF2S1", "EIF2S2",
        "EIF2S3", "EIF3A", "EIF3B", "EIF3C", "EIF3D", "EIF4E", "EIF4G1",
        "EIF4A1", "EEF1A1", "EEF1B2", "EEF2", "RPL3", "RPL5", "RPL7",
        "RPL11", "RPL27", "RPL36", "RPL38", "RPS3", "RPS6", "RPS9",
        "PSMA1", "PSMA2", "PSMA3", "PSMA4", "PSMA5", "PSMA6", "PSMA7",
        "PSMB1", "PSMB2", "PSMB3", "PSMB4", "PSMB5", "PSMB6", "PSMB7",
        "PSMC1", "PSMC2", "PSMC3", "PSMC4", "PSMC5", "PSMC6",
        "PSMD1", "PSMD2", "PSMD3", "PSMD4", "PSMD5", "PSMD6", "PSMD7",
        "PSMD8", "PSMD9", "PSMD10", "PSMD11", "PSMD12", "PSMD13",
        "PSMD14", "UBE2D1", "UBE2N", "CUL1", "CUL3", "RBX1", "GAPDH",
        "PKM", "LDHA", "HK1", "CAD", "RAB5A", "RAB7A", "RAB34", "VAMP3",
        "STX4", "CLTC", "LYST", "NUP62", "NUP98", "NUP153", "NUP214",
        "KPNA2", "KPNB1", "SNRNP70", "SNRPD3", "SNRPA1", "PRPF31",
        "EIF4A3", "XAB2", "POLR2C", "HSP90AA1", "HSPA8", "CCT4", "NACA",
        "PRKAA1", "PRKAA2", "PRKAB1", "PRKAB2", "PRKAG1", "PRKAG2",
        "MTOR", "SGK2", "MAST4", "TAOK3", "POLA1", "POLE", "POLL",
        "RPA1", "WTAP", "GTPBP4", "TUBB3", "ATP6V1B2"
    ],
    "antiviral": [
        "STAT1", "IL17RA", "IL17RE", "IL26", "CXCL13", "XCL1", "CCL21",
        "FPR1", "HRK", "BEX3", "RIPK3", "TNFRSF13C"
    ],
    None: [
        "PRKDC", "XRCC5", "XRCC6", "RAD50", "HES5", "ETS2", "ESRRA",
        "NFATC3", "FHL2", "HNF1A", "TCF7L2", "MAFK", "PA2G4", "NAA15",
        "TAB1", "PPP2R1A", "PRKAR1A", "PRKAR1B", "LATS2", "SLC40A1",
        "TXN2", "NEU1", "BUB1B", "NEK11", "KAT5", "SEPTIN6", "SEPTIN7",
        "SEPTIN9", "KIF17", "TMPRSS9", "PRSS41", "PTPN23", "MYLIP",
        "PLIN3", "FASTK", "RXFP4", "GP1BA", "OGFR", "HCRTR2", "PROKR1",
        "GPR83", "GPR132", "GPR20", "P2RY1", "SUCNR1", "TAAR9", "HTR4",
        "HTR3A", "OR4D1", "OR2H2", "VN1R1", "TRPC3", "KCNF1", "B2M",
        "DDX3X", "TSG101", "USP11", "WNK1", "MAOB", "MS4A7", "PI4KB",
        "WNK4", "IL1RAPL1", "LBR", "DPF1", "FES", "HCK", "PEAK1", "LCK"
    ]
}

In [3]:
# Load the .txt file with confirmed/reliable VACV WR host factor
vacv_hf_path = (
    "/Users/jacobanter/Documents/Code/VACV_screen/Positive_unlabeled_"
    "learning/confirmed_VACV_host_factors.txt"
)

# Bear in mind that in the context of working with files, the `with`
# context manager is preferred as it automatically takes care of closing
# files, even in case of errors/exceptions
with open(vacv_hf_path, "r") as f:
    vacv_host_factors = f.readlines()

# Remove the newline character for all but the last host factor
last_idx = len(vacv_host_factors) - 1

vacv_host_factors = [
    host_factor[:-1] if i != last_idx
    else host_factor
    for i, host_factor in enumerate(vacv_host_factors)
]

In [4]:
# Quickly verify that the dictionary length equals the number of VACV
# host factors (strictly speaking, the number of elements in the
# dictionary values is compared)
n_dict_vals = len([
    val_element
    for dict_val in hf_role_dict.values()
    for val_element in dict_val
])

assert n_dict_vals == len(vacv_host_factors), (
    "A role is not assigned to every VACV host factor!"
)

In [5]:
# Also verify that each VACV host factor is covered
dict_vals = [
    val_element
    for dict_val in hf_role_dict.values()
    for val_element in dict_val
]

assert dict_vals.sort() == vacv_host_factors.sort(), (
    "Not every VACV host factor is assigned a category!"
)

In [6]:
# Now, create a TSV file assigning each VACV host factor to its category
category_list = [
    "proviral" if (host_factor in hf_role_dict["proviral"])
    else "antiviral" if (host_factor in hf_role_dict["antiviral"])
    else None
    for host_factor in vacv_host_factors
]

hf_with_category_df = pd.DataFrame(data={
    "host_factor": vacv_host_factors,
    "category": category_list
})

In [7]:
# Save the TSV file to disk
hf_with_category_df.to_csv(
    "confirmed_VACV_host_factors_with_categories.tsv",
    sep="\t",
    index=False
)

In [8]:
print(hf_with_category_df.loc[
    hf_with_category_df["host_factor"] == "B2M",
    "category"
].values[0])

None
